# 🎬 Colab-Comerza  📦 **v2.0**

Convierte un diseño (JSON) en video animado 9:16 con locución y música.

---
### ⚡ GPU + Run all
`Runtime → Change runtime type → T4 GPU` → `Runtime → Run all`

In [ ]:
# 1. SETUP
import os, sys, json, re, io, base64, textwrap
REPO = "https://github.com/alexdechile/colab-comerza"
if not os.path.exists("colab-comerza"):
    !git clone {REPO}
%cd colab-comerza
%pip install gtts moviepy pillow
!wget "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat-Regular.ttf" -O Montserrat-Regular.ttf 2>&1
!wget "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat-Bold.ttf" -O Montserrat-Bold.ttf 2>&1
!wget "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat-ExtraBold.ttf" -O Montserrat-ExtraBold.ttf 2>&1
import os, glob
for f in glob.glob("Montserrat*.ttf"):
    print(f"  {f}: {os.path.getsize(f)} bytes")
print("Setup listo")


In [ ]:
# 2. IMPORTAR
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from gtts import gTTS
from moviepy.editor import AudioFileClip, ImageSequenceClip, CompositeAudioClip
from moviepy.audio.AudioClip import AudioArrayClip
import torch
print("Colab-Comerza v2.0")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO")


In [ ]:
# 3. CARGAR DISEÑO
with open("mi-diseno.json") as f:
    design = json.load(f)

CANVAS_W = design['canvasSize']['width']   # 595
CANVAS_H = design['canvasSize']['height']  # 842
# Mitad del tamaño original: 540x960
OUT_W, OUT_H = 540, 960
SCALE = OUT_W / CANVAS_W
# Boost de fuente para que sea legible en 540x960
FONT_BOOST = 1.8

def strip_html(t):
    return re.sub(r'<[^>]+>', '', t).strip()

def parse_color(c):
    if c.startswith('#'):
        c = c.lstrip('#')
        if len(c) == 6:
            return tuple(int(c[i:i+2], 16) for i in (0,2,4)) + (255,)
    m = re.findall(r'[\d.]+', c)
    if m: return tuple(int(float(x)) for x in m[:3]) + (255,)
    return (0,0,0,255)

text_elements = []
img_info = None

for el in design['elements']:
    if el['type'] == 'image':
        img_info = {
            'x': el['x'] * SCALE, 'y': el['y'] * SCALE,
            'w': el['width'] * SCALE, 'h': el['height'] * SCALE,
        }
    elif el['type'] == 'text':
        content = strip_html(el['content'])
        if not content: continue
        sy = el.get('scaleY', 1.0) or 1.0
        base_size = el['fontSize'] * SCALE * sy * FONT_BOOST
        text_elements.append({
            'content': content,
            'x': el['x'] * SCALE,
            'y': el['y'] * SCALE,
            'font_size': max(int(base_size), 24),
            'color': el['color'],
            'shadow': el.get('hasShadow', False),
            'sox': el.get('shadowOffsetX', 0) * SCALE,
            'soy': el.get('shadowOffsetY', 0) * SCALE,
            'sc': el.get('shadowColor', 'rgba(0,0,0,0.5)'),
            'align': el.get('alignment', 'left'),
            'width': el.get('width', CANVAS_W) * SCALE,
            'weight': el.get('fontWeight', 400),
        })
        print(f"  '{content}' | size={text_elements[-1]['font_size']}px @ ({int(text_elements[-1]['x'])},{int(text_elements[-1]['y'])})")
print(f"Cargados: {len(text_elements)} textos, escala={SCALE:.2f}, boost={FONT_BOOST}")


In [ ]:
# 4. FONDO
bg = Image.open("imagen.png").convert("RGBA")
iw = int(img_info['w']) if img_info else OUT_W
ih = int(img_info['h']) if img_info else OUT_H
bg_resized = bg.resize((iw, ih), Image.LANCZOS)
base = Image.new("RGBA", (OUT_W, OUT_H), (255,255,255,255))
if img_info:
    base.paste(bg_resized, (int(img_info['x']), int(img_info['y'])), bg_resized)
print(f"Fondo: {iw}x{ih}")


In [ ]:
# 5. FUENTES
font_cache = {}

def get_font(weight, size):
    if weight >= 800:
        ps = ['Montserrat-ExtraBold.ttf', 'Montserrat-Bold.ttf', 'Montserrat-Regular.ttf']
    elif weight >= 700:
        ps = ['Montserrat-Bold.ttf', 'Montserrat-ExtraBold.ttf', 'Montserrat-Regular.ttf']
    else:
        ps = ['Montserrat-Regular.ttf', 'Montserrat-Bold.ttf', 'Montserrat-ExtraBold.ttf']
    k = (tuple(ps), size)
    if k in font_cache: return font_cache[k]
    for p in ps:
        try:
            f = ImageFont.truetype(p, size)
            font_cache[k] = f
            return f
        except: continue
    try:
        f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size)
        font_cache[k] = f; return f
    except:
        f = ImageFont.load_default()
        font_cache[k] = f; return f
print('Fuentes OK')


In [ ]:
# 6. TTS
print("Generando locucion...")
tts_data = []
timing_starts = [0.0, 3.0]

for i, te in enumerate(text_elements):
    text = te['content']
    print(f"  TTS: {text}")
    tts = gTTS(text, lang="es", tld="com.mx", slow=False)
    path = f"tts_{i}.mp3"
    tts.save(path)
    clip = AudioFileClip(path)
    dur = clip.duration
    start = timing_starts[i] if i < len(timing_starts) else (tts_data[-1]['end'] + 0.5)
    tts_data.append({'text': text, 'start': start, 'dur': dur, 'end': start + dur, 'clip': clip})
    print(f"    {dur:.2f}s @ {start:.1f}s")

total_dur = max(14.0, max(t['end'] for t in tts_data) + 0.5)
total_dur = min(total_dur, 14.0)
print(f"Duracion: {total_dur:.1f}s")


In [ ]:
# 7. MUSICA
print("Generando musica...")
SR = 44100
n = int(total_dur * SR)
t = np.linspace(0, total_dur, n, endpoint=False)
pad = (np.sin(2*np.pi*261.63*t) + np.sin(2*np.pi*329.63*t) + np.sin(2*np.pi*392.00*t)) / 3 * 0.3
bass = np.sin(2*np.pi*130.81*t) * 0.15
beat = np.zeros(n)
for b in range(0, n, int(SR * 0.25)):
    beat[b:min(b+800, n)] = 0.4 * np.exp(-np.linspace(0, 5, min(800, n-b)))
music_arr = np.clip(pad + bass + beat, -1, 1) * 0.5
try:
    music_audio = AudioArrayClip(music_arr.reshape(-1, 1), fps=SR).set_duration(total_dur)
except:
    print("  fallback WAV...")
    import wave
    with wave.open("music_bg.wav", "w") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SR)
        wf.writeframes((music_arr * 32767).astype(np.int16).tobytes())
    music_audio = AudioFileClip("music_bg.wav").set_duration(total_dur)
print("Musica lista")


In [ ]:
# 8. RENDER
print("Renderizando frames...")
FPS = 24
n_frames = int(total_dur * FPS)
frames = []
FADE_DUR = 0.35
SLIDE_PX = 30

def render_text(draw, te, opacity, slide_off):
    font = get_font(te['weight'], te['font_size'])
    content = te['content']
    # Word-wrap long text
    avg_char_w = font.getbbox("A")[2] - font.getbbox("A")[0]
    max_chars = int(te['width'] / avg_char_w) if avg_char_w > 0 else 20
    lines = textwrap.wrap(content, width=max(max_chars, 10))
    if not lines:
        lines = [content]
    line_h = font.getbbox('Ag')[3] - font.getbbox('Ag')[1] + 8
    total_h = len(lines) * line_h
    start_y = te['y'] + slide_off - (total_h / 2)
    color = parse_color(te['color'])
    color = tuple(int(c * opacity) for c in color[:3]) + (255,)
    sc = parse_color(te['sc'])
    sc = tuple(int(c * opacity) for c in sc[:3]) + (int(sc[3] * opacity),)
    for li, line in enumerate(lines):
        bbox = font.getbbox(line)
        lw = bbox[2] - bbox[0]
        x = te['x']
        if te['align'] == 'center':
            x += (te['width'] - lw) / 2
        y = start_y + li * line_h
        if te['shadow']:
            draw.text((x + te['sox'], y + te['soy']), line, fill=sc, font=font)
        draw.text((x, y), line, fill=color, font=font)

for fi in range(n_frames):
    t = fi / FPS
    frame = base.copy()
    draw = ImageDraw.Draw(frame)
    for te, td in zip(text_elements, tts_data):
        if td['start'] <= t <= td['end']:
            if t < td['start'] + FADE_DUR:
                prog = (t - td['start']) / FADE_DUR
                opacity, slide_off = prog, SLIDE_PX * (1 - prog)
            else:
                opacity, slide_off = 1.0, 0
            render_text(draw, te, opacity, slide_off)
    frames.append(np.array(frame.convert('RGB')))
    if fi % 48 == 0 or fi == n_frames - 1:
        print(f"  {fi+1}/{n_frames} ({100*(fi+1)//n_frames}%)")
print(f"{len(frames)} frames")


In [ ]:
# 9. VIDEO
print("Armando video...")
vc = ImageSequenceClip(frames, fps=FPS)
tracks = [music_audio]
for td in tts_data:
    tracks.append(td['clip'].set_start(td['start']))
fa = CompositeAudioClip(tracks)
vc = vc.set_audio(fa)
OUTPUT = "comerza_animado.mp4"
print("Escribiendo MP4...")
vc.write_videofile(OUTPUT, codec='libx264', audio_codec='aac', fps=FPS, bitrate='2500k', preset='medium', threads=2, logger=None)
print("Video listo:", OUTPUT)


In [ ]:
# 10. DESCARGAR
from google.colab import files
files.download(OUTPUT)
